In [0]:
# %sql
# DELETE FROM workspace.default.fx_bronze WHERE date < (SELECT MAX(date) FROM workspace.default.`fx-boe-data`);

# INSERT INTO workspace.default.fx_bronze
# SELECT * FROM workspace.default.`fx-boe-data`
# WHERE date >= (SELECT MAX(date) FROM workspace.default.fx_bronze);

Bronze -> Silver -> Gold

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("fx_boe_data")

# BoE returns dates as "02 Jan 2024" — a plain cast to date silently returns
# null on this format. to_date() needs the explicit format string.
silver_df = bronze_df.withColumn(
    "rate_date", F.to_date(F.col("date"), "dd MMM yyyy")
)

# BoE uses ".." for missing values. Cast to string BEFORE checking for "..",
# then to double — comparing a numeric column directly to ".." throws an
# error instead of nulling it.
rate_cols = {
    "XUDLUSS": "gbp_usd", "XUDLERS": "gbp_eur", "XUDLJYS": "gbp_jpy",
    "XUDLCDS": "gbp_cad", "XUDLSFS": "gbp_chf", "XUDLADS": "gbp_aud",
    "XUDLSGS": "gbp_sgd", "XUDLSKS": "gbp_sek", "XUDLDKS": "gbp_dkk",
    "IUDBEDR": "bank_rate", "IUDSOIA": "sonia",
}
for raw_col, clean_name in rate_cols.items():
    silver_df = silver_df.withColumn(
        clean_name,
        F.when(F.col(raw_col).cast("string") == "..", None)
         .otherwise(F.col(raw_col).cast("double"))
    )

silver_df = silver_df.select("rate_date", *rate_cols.values()).dropDuplicates(["rate_date"])
silver_df = silver_df.filter(F.col("rate_date").isNotNull())

# --- Quality gates ---
n_total = silver_df.count()
n_null_dates = bronze_df.filter(F.to_date(F.col("date"), "dd MMM yyyy").isNull()).count()
n_bad_bank_rate = silver_df.filter((F.col("bank_rate") < 0) | (F.col("bank_rate") > 20)).count()

assert n_total > 0, "Silver table is empty — check date parsing before going further."
assert n_bad_bank_rate == 0, f"{n_bad_bank_rate} rows have an impossible Bank Rate — likely a parsing bug, not real data."
print(f"Silver: {n_total} rows, {n_null_dates} rows dropped for unparseable dates.")

silver_df.write.format("delta").mode("overwrite").saveAsTable("fx_silver")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date, timedelta

silver_df = spark.table("fx_silver")
w = Window.orderBy("rate_date")

fx_cols = ["gbp_usd", "gbp_eur", "gbp_jpy", "gbp_cad", "gbp_chf", "gbp_aud", "gbp_sgd", "gbp_sek", "gbp_dkk"]

gold_df = silver_df
for col in fx_cols:
    gold_df = gold_df.withColumn(
        f"{col}_logret",
        F.log(F.col(col) / F.lag(F.col(col)).over(w))
    )

gold_df = gold_df.orderBy("rate_date")

latest = gold_df.agg(F.max("rate_date")).collect()[0][0]
if latest < date.today() - timedelta(days=10):
    print(f"WARNING: gold table is stale — latest date is {latest}")

gold_df.write.format("delta").mode("overwrite").saveAsTable("fx_gold")
print(f"Gold: {gold_df.count()} rows, {latest} is the most recent date.")

Bronze stage: Duplicate raw BoE FX data to a bronze table for initial ingestion.
(See Cell 1: commented SQL CREATE TABLE AS SELECT)

Silver stage: Clean and transform bronze data.
- Parse dates with explicit format.
- Replace ".." with nulls and cast FX rates to double.
- Rename columns for clarity.
- Drop duplicate and null dates.
- Quality checks: count rows, dropped null dates, and invalid bank rates.
- Save cleaned data to fx_silver Delta table.

Gold stage: Calculate log returns for FX rates.
- Compute log returns for each FX rate column.
- Order by date.
- Freshness check: warn if latest date is >10 days old.
- Save enriched data to fx_gold Delta table.